<a href="https://colab.research.google.com/github/pullz6/Large-Scale-Fraud-Detection-System/blob/main/Data_preprocessing_Model_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install mlflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 733.8/733.8 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.2/196.2 kB 10.2 MB/s eta 0:00:00


In [23]:
!pip install --upgrade mlflow>=2.3.0

In [24]:
!pip install pyngrok -q

In [25]:
def set_mlflow():
  get_ipython().system_raw("mlflow ui --port 5000 &")
  ngrok.kill()
  NGROK_AUTH_TOKEN = "2slIw2XdGuFjpkotlKi9OYKkNxH_5dr7FtkWh4qMxea6sUoat"
  ngrok.set_auth_token(NGROK_AUTH_TOKEN)
  ngrok_tunnel = ngrok.connect(addr="5000", proto="http", bind_tls=True)
  print("MLflow Tracking UI:", ngrok_tunnel.public_url)
  return ngrok_tunnel.public_url

In [2]:
!pip install pyspark

In [3]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 23.7 MB/s eta 0:00:00


In [4]:
!pip install -q kaggle

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
import random

import os
from google.colab import drive
from kaggle.api.kaggle_api_extended import KaggleApi
import json
from pyngrok import ngrok, conf
import mlflow

import pyspark
from pyspark.sql import Row
from datetime import datetime, timedelta

## Load Data from Kaggle

In [7]:
#Read creds
drive.mount('/content/drive')

#loading all the required datasets
creds_path = ('/content/drive/MyDrive/Projects/Creds/kaggle.json')

Mounted at /content/drive


In [8]:
with open(creds_path, 'r') as f:
    creds = json.load(f)

In [9]:
!mkdir -p ~/.kaggle

In [10]:
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(creds, f)

In [11]:
# Set permissions
!chmod 600 ~/.kaggle/kaggle.json

# Verify setup
!kaggle datasets list -s "ieee fraud"

# STEP 2: DOWNLOAD DATASET
# Download the dataset (this is a known mirror of the original competition data)
!kaggle datasets download -d kartik2112/fraud-detection

# Unzip the files (creates a 'fraud-detection' folder)
!unzip fraud-detection.zip -d fraud-detection

ref                                                        title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------------------  -------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
mlg-ulb/creditcardfraud                                    Credit Card Fraud Detection                          69155672  2018-03-23 01:17:27.913000         901650      12239  0.85294116       
muhakabartay/yourallmodelsdata                             IEEE-CIS Fraud Detection Models Data                 29768007  2019-09-18 07:57:04.473000            684         17  1.0              
whenamancodes/fraud-detection                              Fraud Detection                                      69155672  2022-09-12 11:54:40.550000          10600        117  1.0              
kyakovlev/ieee-submissions-and

In [12]:
# List downloaded files
!ls -la fraud-detection/

total 489852
drwxr-xr-x 2 root root      4096 Jun 15 11:17 .
drwxr-xr-x 1 root root      4096 Jun 15 11:17 ..
-rw-r--r-- 1 root root 150354339 Aug  5  2020 fraudTest.csv
-rw-r--r-- 1 root root 351238196 Aug  5  2020 fraudTrain.csv


## Load for training from pandas

In [61]:
train = pd.read_csv("fraud-detection/fraudTrain.csv")
test = pd.read_csv("fraud-detection/fraudTest.csv")

In [62]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  object 
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  object 
 4   category               1296675 non-null  object 
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  object 
 7   last                   1296675 non-null  object 
 8   gender                 1296675 non-null  object 
 9   street                 1296675 non-null  object 
 10  city                   1296675 non-null  object 
 11  state                  1296675 non-null  object 
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long              

In [63]:
train.drop('Unnamed: 0',inplace=True,axis=1)

In [ ]:
test['is_fraud'].value_counts()

In [79]:
print("Class Distribution:")
print(train['is_fraud'].value_counts())
print(f"Percentage: \n{train['is_fraud'].value_counts(normalize=True) * 100}")

# Imbalance ratio (%)
imbalance_ratio = train['is_fraud'].value_counts().min() / train['is_fraud'].value_counts().max()
print(f"Imbalance ratio: {imbalance_ratio:.3f}")

Class Distribution:
is_fraud
0    1290118
1       7557
Name: count, dtype: int64
Percentage: 
is_fraud
0    99.417651
1     0.582349
Name: proportion, dtype: float64
Imbalance ratio: 0.006


In [82]:
# Split instances into majority vs minority class/classes
df_majority = train[train['is_fraud'] == 0]
df_minority = train[train['is_fraud'] == 1]

# Undersampling: we keep as many majority instances (n) as minority ones
df_majority_downsampled = df_majority.sample(n=len(df_minority),random_state=42)
df_balanced = pd.concat([df_majority_downsampled, df_minority])

print(f"Original dataset: {len(train)}")
print(f"Balanced dataset: {len(df_balanced)}")

Original dataset: 1297675
Balanced dataset: 15114


In [85]:
df_balanced.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15114 entries, 204293 to 959
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trans_date_trans_time  15062 non-null  object 
 1   cc_num                 15114 non-null  int64  
 2   merchant               15114 non-null  object 
 3   category               15114 non-null  object 
 4   amt                    15114 non-null  object 
 5   first                  15114 non-null  object 
 6   last                   15114 non-null  object 
 7   gender                 15114 non-null  object 
 8   street                 15114 non-null  object 
 9   city                   15114 non-null  object 
 10  state                  15114 non-null  object 
 11  zip                    15114 non-null  object 
 12  lat                    15114 non-null  object 
 13  long                   15114 non-null  float64
 14  city_pop               15114 non-null  float64
 15  job 

## Archive

In [39]:
from pyspark.sql import SparkSession

In [40]:
spark = SparkSession.builder.appName("IEEE_Fraud_Detection").config("spark.driver.memory", "8g").config("spark.executor.memory", "8g").getOrCreate()

In [41]:
# Load the training and testing data
test = spark.read.csv(
    f"fraud-detection/fraudTest.csv",
    header=True,
    inferSchema=True
)

train = spark.read.csv(
    f"fraud-detection/fraudTrain.csv",
    header=True,
    inferSchema=True
)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# DataFrame approach
print(f"Training transactions: {train.count():,} rows")
train.printSchema()

# Show some statistics
train.describe().show()

In [ ]:
test.describe().show()

In [ ]:
fake = Faker()

def generate_fake_fraud_data(num_records):
    data = []
    for _ in range(num_records):
        # Generate base transaction data
        cc_num = fake.credit_card_number()
        merchant = fake.company()
        category = random.choice(['entertainment', 'food_dining', 'gas_transport',
                                 'grocery_net', 'grocery_pos', 'health_fitness',
                                 'home', 'kids_pets', 'misc_net', 'misc_pos',
                                 'personal_care', 'shopping_net', 'shopping_pos',
                                 'travel'])
        amt = round(random.uniform(1, 1000), 2)

        # Generate personal information
        gender = random.choice(['M', 'F'])
        first = fake.first_name_male() if gender == 'M' else fake.first_name_female()
        last = fake.last_name()

        # Generate location data
        street = fake.street_address()
        city = fake.city()
        state = fake.state_abbr()
        zip_code = fake.zipcode()
        lat = float(fake.latitude())
        long = float(fake.longitude())
        city_pop = random.randint(1000, 1000000)

        # Generate employment data
        job = fake.job()

        # Transaction identifiers
        trans_num = fake.uuid4()
        unix_time = int(fake.date_time_this_year().timestamp())

        # Merchant location (slightly different from customer)
        merch_lat = lat + random.uniform(-0.1, 0.1)
        merch_long = long + random.uniform(-0.1, 0.1)

        # Fraud determination (5% base fraud rate)
        is_fraud = 0
        if random.random() < 0.05:
            is_fraud = 1
            # Make fraudulent transactions look different
            amt = round(random.uniform(500, 5000), 2)  # Higher amounts
            category = random.choice(['misc_net', 'shopping_net', 'travel'])  # Common fraud categories
            merch_lat = lat + random.uniform(-1, 1)  # More distant from customer
            merch_long = long + random.uniform(-1, 1)

        data.append(Row(
            _c0=len(data) + 1,  # Sequential ID
            cc_num=cc_num,
            merchant=merchant,
            category=category,
            amt=amt,
            first=first,
            last=last,
            gender=gender,
            street=street,
            city=city,
            state=state,
            zip=zip_code,
            lat=lat,
            long=long,
            city_pop=city_pop,
            job=job,
            trans_num=trans_num,
            unix_time=unix_time,
            merch_lat=merch_lat,
            merch_long=merch_long,
            is_fraud=is_fraud
        ))
    return data

# Generate 1000 fake records
fake_data = generate_fake_fraud_data(1000)

# Convert to Spark DataFrame
fake_df = spark.createDataFrame(fake_data)
fake_df.show(5)

In [ ]:
train = train.unionByName(fake_df, allowMissingColumns=True)

In [ ]:
train.describe().show()

In [ ]:
train.printSchema()

In [ ]:
uri = set_mlflow()
mlflow.set_tracking_uri(uri=uri)

In [74]:
train = pd.concat([train, fake_df])